In [1]:
# General
import numpy as np
import pandas as pd
import os
import requests

import urllib.request, json 
from tqdm import tqdm

In [2]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [3]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Prepare PUMS Variables Mapping files by Year

In [4]:
url_to_import = "https://api.census.gov/data/2021/cps/foodsec/dec/variables.json"

with urllib.request.urlopen(url_to_import) as url:
    dict_cps = json.load(url)



In [5]:
years = range(2009, 2023)
years

range(2009, 2023)

In [6]:
# dict_cps['variables']['HRHHID']
# dict_cps['variables']['HRHTYPE']

In [17]:
# initialize empty list to store data frames
# iterate through each year
    # pull PUMS variables list from json file found on ACS website
    # convert to dictionary
    # convert to pandas data frame
    # apply year tag
    # append to list
# concatenate all data frames together

list_df_cps = []

for year in tqdm(years):
    try:
        if year in [1995, 1997, 1999]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/apr/variables.json"
        if year in [1998]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/aug/variables.json"
        if year in [2000]:
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/sep/variables.json"
        if year in sequence(2001, 2022, 1):
            url_to_import = f"https://api.census.gov/data/{year}/cps/foodsec/dec/variables.json"
            
        with urllib.request.urlopen(url_to_import) as url:
    
            dict_cps = json.load(url)
    
            # convert to pandas data frame
            df_cps = pd.DataFrame.from_dict(dict_cps['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
            df_cps['Year'] = year
            list_df_cps.append(df_cps)
    except Exception as e: print(e)
    
df_cps_raw = pd.concat(list_df_cps)
print(df_cps_raw.shape)
df_cps_raw.head()

100%|██████████| 14/14 [00:30<00:00,  2.15s/it]

(7012, 11)


,ID,label,concept,predicateType,group,limit,predicateOnly,suggested-weight,values,is-weight,Year
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN,2009
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN,2009
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,NaN,NaN,NaN,2009
3,PEEDUCA,Demographics-highest level of school completed,NaN,int,N/A,0,NaN,PWSSWGT,"{'item': {'46': 'DOCTORATE DEGREE(EX:PhD,EdD)'...",NaN,2009
4,PUBUS1,Labor Force-unpaid work in family business/far...,NaN,int,N/A,0,NaN,PWCMPWGT,"{'item': {'-3': 'Refused', '-1': 'Blank', '-2'...",NaN,2009


In [21]:
# df_cps_raw[df_cps_raw['ID'] == 'HRHHID']
df_cps_raw[(df_cps_raw['predicateType'] == 'int') & df_cps_raw['values'].isna()]

,ID,label,concept,predicateType,group,limit,predicateOnly,suggested-weight,values,is-weight,Year
35,PWSUPWGT,Weight - person weight for supplement household,NaN,int,N/A,0,NaN,NaN,NaN,True,2009
124,HHSUPWGT,Weight - household weight for supplement house...,NaN,int,N/A,0,NaN,NaN,NaN,True,2009
139,HESP9TC,Program - topcode flag for number getting WIC ...,NaN,int,N/A,0,NaN,HHSUPWGT,NaN,NaN,2009
216,PWVETWGT,Weight-veterans weight,NaN,int,N/A,0,NaN,NaN,NaN,True,2009
218,PWORWGT,Weight-outgoing rotation weight,NaN,int,N/A,0,NaN,NaN,NaN,True,2009
...,...,...,...,...,...,...,...,...,...,...,...
444,HWHHWGT,Weight-household,NaN,int,N/A,0,NaN,NaN,NaN,True,2022
468,HES1B,"Expend shopped at dollar stores, pharmacies, etc.",NaN,int,N/A,0,NaN,HHSUPWGT,NaN,NaN,2022
469,HES1A,Expend shopped at supermarket/grocery store la...,NaN,int,N/A,0,NaN,HHSUPWGT,NaN,NaN,2022
480,HES1C,Expend bought food at restaurant/cafeteria/etc...,NaN,int,N/A,0,NaN,HHSUPWGT,NaN,NaN,2022


In [68]:
# remove variables that don't need a variable to value mapping
# df_cps = df_cps_raw.dropna(subset=["values"]).reset_index(drop = True)
df_cps = df_cps_raw.copy()
print(df_cps.shape[0])
df_cps

7012


,ID,label,concept,predicateType,group,limit,predicateOnly,suggested-weight,values,is-weight,Year
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN,2009
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN,2009
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,NaN,NaN,NaN,2009
3,PEEDUCA,Demographics-highest level of school completed,NaN,int,N/A,0,NaN,PWSSWGT,"{'item': {'46': 'DOCTORATE DEGREE(EX:PhD,EdD)'...",NaN,2009
4,PUBUS1,Labor Force-unpaid work in family business/far...,NaN,int,N/A,0,NaN,PWCMPWGT,"{'item': {'-3': 'Refused', '-1': 'Blank', '-2'...",NaN,2009
...,...,...,...,...,...,...,...,...,...,...,...
507,H_ID_PL,HH PP Unique ID,NaN,string,N/A,0,NaN,NaN,NaN,NaN,2022
508,PREMP,Labor Force-employed non-farm/non-private hhld...,NaN,int,N/A,0,NaN,PWCMPWGT,{'item': {'1': 'Employed Persons (excluding ag...,NaN,2022
509,PTWK,Earnings-weekly-top code flag,NaN,int,N/A,0,NaN,PWORWGT,"{'item': {'1': 'Topcoded', '0': 'Not Topcoded'}}",NaN,2022
510,PUNLFCK2,Labor Force-outgoing rotation filter,NaN,int,N/A,0,NaN,PWCMPWGT,"{'item': {'1': 'MIS-CK=4, Goto NLFJH', '-2': '...",NaN,2022


In [ ]:
# list_keys = []

# for dict in df_cps['values'].values:
#     list_keys.append(list(dict.keys())[0])

# unique(list_keys)

In [69]:
# create empty list to store data frames
# iterate through each year
    # create empty list to store data frames
    # subset all variables to year
        # create empty list to store data frames
            # subset pums variables to one ID at a time
            # iterate through all key/value combinations in dictionaries that represent the value mappings to make pandas data frames 
            # store them in list of data frames
        # concatenate specific ID variable mappings together
        # add some labels, clean column names
    # apply year tag
    # convert to pandas data frame
    # store in list of data frames

# concatenate all data frames together

list_df_years = []

for year in tqdm(years):
    try:
        df_cps_vars = df_cps[df_cps['Year'] == year]
        
        list_df = []
        
        for ID in df_cps_vars['ID'].values:
            
            df_ID = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)

            if df_ID['values'][0] is np.nan:
                df_vars['Value1'] = np.nan
                df_vars['Value2'] = np.nan
                df_vars['Description'] = np.nan
                df_vars['ID'] = ID
                df_vars['Label'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)['label'].values[0]
                df_vars['Suggested Weight'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)['suggested-weight'].values[0]
                df_vars = df_vars[['Label', 'ID', 'Value1', 'Value2', 'Description', 'Suggested Weight']]

            else:
            
                for key in list(df_ID['values'][0].keys()):
                    
                    if key == 'item':
                        dict_values = {
                                         'Value1'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                       , 'Value2'     : list(list(df_ID['values'].values)[0]['item'].keys()  )
                                       , 'Description': list(list(df_ID['values'].values)[0]['item'].values())
                                      }
                        df_vars = pd.DataFrame(dict_values)
                        
                        
                    if key == 'range':
                        
                        list_range = []
            
                        for value in df_ID['values'][0]['range']:
                            dict_values = {
                                            'Value1'     : [value['min']]
                                          , 'Value2'     : [value['max']]
                                          , 'Description': [value['description']]
                                         }
                            
                            list_range.append(pd.DataFrame(dict_values))
                            
                        df_vars = pd.concat(list_range)
                        
                    df_vars['ID'] = ID
                    df_vars['Label'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)['label'].values[0]
                    df_vars['Suggested Weight'] = df_cps_vars[df_cps_vars['ID'] == ID].reset_index(drop = True)['suggested-weight'].values[0]
            
                    df_vars = df_vars[['Label', 'ID', 'Value1', 'Value2', 'Description', 'Suggested Weight']]
            
            list_df.append(df_vars)
        
        
        df_cps_vars = pd.concat(list_df)
        df_cps_vars['Year'] = year
        
        list_df_years.append(df_cps_vars)
    except Exception as e: print(e)

df_cps_vars = pd.concat(list_df_years)

100%|██████████| 14/14 [00:18<00:00,  1.34s/it]


In [14]:
# df_cps = df_cps_raw.copy()
# df_cps = df_cps[df_cps['values'].isna()]
# df_cps = df_cps[['label', 'ID', 'concept', 'suggested-weight', 'Year']]
# df_cps = df_cps.rename(columns = {'label':'Label', 'concept':'Description', 'suggested-weight':'Suggested Weight'})
# df_cps['Value1'] = np.nan
# df_cps['Value2'] = np.nan
# df_cps

,Label,ID,Description,Suggested Weight,Year,Value1,Value2
0,Census API FIPS 'for' clause,for,Census API Geography Specification,NaN,2009,NaN,NaN
1,Census API FIPS 'in' clause,in,Census API Geography Specification,NaN,2009,NaN,NaN
2,Uniform Census Geography Identifier clause,ucgid,Census API Geography Specification,NaN,2009,NaN,NaN
35,Weight - person weight for supplement household,PWSUPWGT,NaN,NaN,2009,NaN,NaN
124,Weight - household weight for supplement house...,HHSUPWGT,NaN,NaN,2009,NaN,NaN
...,...,...,...,...,...,...,...
468,"Expend shopped at dollar stores, pharmacies, etc.",HES1B,NaN,HHSUPWGT,2022,NaN,NaN
469,Expend shopped at supermarket/grocery store la...,HES1A,NaN,HHSUPWGT,2022,NaN,NaN
480,Expend bought food at restaurant/cafeteria/etc...,HES1C,NaN,HHSUPWGT,2022,NaN,NaN
496,Weight-family,PWFMWGT,NaN,NaN,2022,NaN,NaN


In [70]:
# Sort variable mapping
df_cps_vars = df_cps_vars.sort_values(['Year', 'ID', 'Value1'], ascending = [False, True, True])
df_cps_vars

,Label,ID,Value1,Value2,Description,Suggested Weight,Year
6,Geographical Division,GEDIV,1,1,NEW ENGLAND,PWCMPWGT,2022
8,Geographical Division,GEDIV,2,2,MIDDLE ATLANTIC,PWCMPWGT,2022
0,Geographical Division,GEDIV,3,3,EAST NORTH CENTRAL,PWCMPWGT,2022
1,Geographical Division,GEDIV,4,4,WEST NORTH CENTRAL,PWCMPWGT,2022
3,Geographical Division,GEDIV,5,5,SOUTH ATLANTIC,PWCMPWGT,2022
...,...,...,...,...,...,...,...
0,Uniform Census Geography Identifier clause,ucgid,NaN,NaN,NaN,NaN,2009
1,Uniform Census Geography Identifier clause,ucgid,NaN,NaN,NaN,NaN,2009
2,Uniform Census Geography Identifier clause,ucgid,NaN,NaN,NaN,NaN,2009
3,Uniform Census Geography Identifier clause,ucgid,NaN,NaN,NaN,NaN,2009


In [74]:
df_cps_vars[df_cps_vars['ID'] == 'HRHHID']

,Label,ID,Value1,Value2,Description,Suggested Weight,Year
0,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2022
1,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2022
2,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2022
3,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2022
4,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2022
...,...,...,...,...,...,...,...
1,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2009
2,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2009
3,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2009
4,"Household-identifier,scrambled",HRHHID,NaN,NaN,NaN,HWHHWGT,2009


In [75]:
# export locally
df_cps_vars.to_excel(os.path.join(path_config, 'CPS Variables Mapping ALL YEARS.xlsx'), index=False)